### Import Packages

In [106]:
from argparse import Namespace
import os
import re
import numpy as np
import pandas as pd
import ast

import plotly.graph_objects as go

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from tqdm.notebook import tqdm

### Vocabulary

In [107]:
class Vocabulary(object):
    def __init__(self, token_to_idx=None):
        if token_to_idx is None:
            token_to_idx = {}
        self._token_to_idx = token_to_idx
        self._idx_to_token = {idx: token for token, idx in self._token_to_idx.items()}

    def add_token(self, token):
        if token in self._token_to_idx:
            index = self._token_to_idx[token]
        else:
            index = len(self._token_to_idx)
            self._token_to_idx[token] = index
            self._idx_to_token[index] = token
        return index

    def lookup_token(self, token):
        return self._token_to_idx[token]

    def lookup_index(self, index):
        if index not in self._idx_to_token:
            raise KeyError("the index (%d) is not in the Vocabulary" % index)
        return self._idx_to_token[index]

    def __len__(self):
        return len(self._token_to_idx)

In [108]:
class SequenceVocabulary(Vocabulary):
    def __init__(self, token_to_idx=None, unk_token="<UNK>",
                 mask_token="<MASK>", begin_seq_token="<BEGIN>",
                 end_seq_token="<END>"):
        super(SequenceVocabulary, self).__init__(token_to_idx)

        self._mask_token = mask_token
        self._unk_token = unk_token
        self._begin_seq_token = begin_seq_token
        self._end_seq_token = end_seq_token

        self.mask_index = self.add_token(self._mask_token)
        self.unk_index = self.add_token(self._unk_token)
        self.begin_seq_index = self.add_token(self._begin_seq_token)
        self.end_seq_index = self.add_token(self._end_seq_token)

    def lookup_token(self, token):
        if self.unk_index >= 0:
            return self._token_to_idx.get(token, self.unk_index)
        else:
            return self._token_to_idx[token]

### Vectorizer

In [109]:
class SentimentVectorizer(object):
    def __init__(self, text_vocab, label_vocab):
        self.text_vocab = text_vocab
        self.label_vocab = label_vocab

    def vectorize(self, text, vector_length=-1):
        tokens = text # tokens are pre-tokenized in line tokens before, write like this to skip cleantext and tokenize
        indices = [self.text_vocab.begin_seq_index]
        indices.extend(self.text_vocab.lookup_token(t) for t in tokens)
        indices.append(self.text_vocab.end_seq_index)

        if vector_length < 0:
            vector_length = len(indices)

        out_vector = np.zeros(vector_length, dtype=np.int64)
        out_vector[:len(indices)] = indices
        out_vector[len(indices):] = self.text_vocab.mask_index

        return out_vector, len(indices)

    @classmethod
    def from_dataframe(cls, df, text_col="text", label_col="label"):
        word_vocab = SequenceVocabulary()
        label_vocab = Vocabulary()

        for _, row in df.iterrows():
            tokens = row[text_col]
            for tok in tokens:
                word_vocab.add_token(tok)
            label_vocab.add_token(row[label_col])

        return cls(word_vocab, label_vocab)

### Dataset

In [110]:
class SentimentDataset(Dataset):
    def __init__(self, df, vectorizer, train_frac=0.7, val_frac=0.15, test_frac=0.15, random_state=42):

        self._vectorizer = vectorizer

        train_df, temp_df = train_test_split(df, test_size=(1-train_frac), random_state=random_state, stratify=df['airline_sentiment'])
        val_df, test_df = train_test_split(temp_df, test_size=test_frac/(test_frac+val_frac),
                                           random_state=random_state, stratify=temp_df['airline_sentiment'])

        train_df = train_df.copy(); train_df['split'] = 'train'
        val_df = val_df.copy(); val_df['split'] = 'val'
        test_df = test_df.copy(); test_df['split'] = 'test'

        self.df = pd.concat([train_df, val_df, test_df], ignore_index=True)

        self._max_seq_length = max(self.df['tokens'].apply(len)) + 2

        self.train_df = train_df.reset_index(drop=True)
        self.train_size = len(self.train_df)

        self.val_df = val_df.reset_index(drop=True)
        self.validation_size = len(self.val_df)

        self.test_df = test_df.reset_index(drop=True)
        self.test_size = len(self.test_df)

        self._lookup_dict = {
            'train': (self.train_df, self.train_size),
            'val': (self.val_df, self.validation_size),
            'test': (self.test_df, self.test_size)
        }

        self.set_split('train')

        class_counts = self.train_df['airline_sentiment'].value_counts().to_dict()

        sorted_counts = sorted(class_counts.items(), key=lambda x: vectorizer.label_vocab.lookup_token(x[0]))
        frequencies = [count for _, count in sorted_counts]

        self.class_weights = 1.0 / torch.tensor(frequencies, dtype=torch.float32)

        self.class_weights = self.class_weights / self.class_weights.sum()

    @classmethod
    def load_dataset_and_make_vectorizer(cls, csv_path):
        df = pd.read_csv(csv_path)
        df['tokens'] = df['tokens'].apply(ast.literal_eval)
        vectorizer = SentimentVectorizer.from_dataframe(df, text_col='tokens', label_col='airline_sentiment')
        return cls(df, vectorizer)

    def get_vectorizer(self):
        return self._vectorizer

    def set_split(self, split="train"):
        self._target_split = split
        self._target_df, self._target_size = self._lookup_dict[split]

    def __len__(self):
        return self._target_size

    def __getitem__(self, index):
        row = self._target_df.iloc[index]
        x_vec, seq_len = self._vectorizer.vectorize(row.tokens, vector_length=self._max_seq_length)
        y_index = self._vectorizer.label_vocab.lookup_token(row.airline_sentiment)
        return {
            'x_data': torch.tensor(x_vec, dtype=torch.long),
            'y_target': torch.tensor(y_index, dtype=torch.long),
            'x_length': seq_len
        }

    def get_num_batches(self, batch_size):
        return len(self) // batch_size


def generate_batches(dataset, batch_size, shuffle=True, drop_last=True, device="cpu"):
    dataloader = DataLoader(dataset=dataset, batch_size=batch_size,
                            shuffle=shuffle, drop_last=drop_last)
    for data_dict in dataloader:
        yield {name: tensor.to(device) for name, tensor in data_dict.items()}

### Models

In [111]:
def column_gather(y_out, x_lengths):
    x_lengths = x_lengths.long().detach().cpu().numpy() - 1
    out = [y_out[batch_index, col_index]
           for batch_index, col_index in enumerate(x_lengths)]
    return torch.stack(out)


def column_summation(y_out, x_lengths, mode="mean"):
    x_lengths = x_lengths.long().detach().cpu().numpy()
    out = []

    for batch_index, length in enumerate(x_lengths):
        valid_vecs = y_out[batch_index, :length, :]
        if mode == "mean":
            pooled = valid_vecs.mean(dim=0)
        elif mode == "max":
            pooled, _ = valid_vecs.max(dim=0)
        else:
            raise ValueError(f"Unknown mode: {mode}")
        out.append(pooled)

    return torch.stack(out)

def get_model(args, num_embeddings, num_classes):
    embedding_size = args.char_embedding_size
    rnn_hidden_size = args.rnn_hidden_size
    padding_idx = 0

    if args.model_type == "RNN":
        model = RNNClassifier(embedding_size, num_embeddings, num_classes,
                              rnn_hidden_size, bidirectional=args.bidirectional)
    elif args.model_type == "GRU":
        model = GRUClassifier(embedding_size, num_embeddings, num_classes,
                              rnn_hidden_size, bidirectional=args.bidirectional)
    elif args.model_type == "LSTM":
        model = LSTMClassifier(embedding_size, num_embeddings, num_classes,
                               rnn_hidden_size, bidirectional=args.bidirectional)
    else:
        raise ValueError(f"Unknown model_type: {args.model_type}")

    return model.to(args.device)

class RNNClassifier(nn.Module):
    def __init__(self, embedding_size, num_embeddings, num_classes,
                 rnn_hidden_size, bidirectional=False, batch_first=True, padding_idx=0, pooling_mode="gather"):
        super(RNNClassifier, self).__init__()
        self.num_directions = 2 if bidirectional else 1

        self.emb = nn.Embedding(num_embeddings=num_embeddings,
                                embedding_dim=embedding_size,
                                padding_idx=padding_idx)

        self.rnn = nn.RNN(input_size=embedding_size,
                          hidden_size=rnn_hidden_size,
                          batch_first=batch_first,
                          bidirectional=bidirectional)

        self.fc1 = nn.Linear(rnn_hidden_size*self.num_directions,
                             rnn_hidden_size*self.num_directions)
        self.fc2 = nn.Linear(rnn_hidden_size*self.num_directions,
                             num_classes)
        self.bn1 = nn.BatchNorm1d(rnn_hidden_size*self.num_directions)
        self.dropout = nn.Dropout(0.2)
        self.pool_mode = pooling_mode

    def forward(self, x_in, x_lengths=None, apply_softmax=False):
        x_embedded = self.emb(x_in)
        y_out, _ = self.rnn(x_embedded)

        if x_lengths is not None:
            if self.pool_mode == "gather":
                y_out = column_gather(y_out, x_lengths)
            elif self.pool_mode == "mean":
                y_out = column_summation(y_out, x_lengths, mode="mean")
            elif self.pool_mode == "max":
                y_out = column_summation(y_out, x_lengths, mode="max")
        else:
            y_out = y_out[:, -1, :]

        y_out = F.relu(self.bn1(self.fc1(self.dropout(y_out))))
        y_out = self.fc2(self.dropout(y_out))

        if apply_softmax:
            y_out = F.softmax(y_out, dim=1)
        return y_out


class GRUClassifier(RNNClassifier):
    def __init__(self, embedding_size, num_embeddings, num_classes,
                 rnn_hidden_size, bidirectional=False, batch_first=True, padding_idx=0, pooling_mode="gather"):
        super(GRUClassifier, self).__init__(embedding_size, num_embeddings, num_classes, rnn_hidden_size, bidirectional, batch_first, padding_idx, pooling_mode)
        self.rnn = nn.GRU(input_size=embedding_size,
                          hidden_size=rnn_hidden_size,
                          batch_first=batch_first,
                          bidirectional=bidirectional)


class LSTMClassifier(RNNClassifier):
    def __init__(self, embedding_size, num_embeddings, num_classes,
                 rnn_hidden_size, bidirectional=False, batch_first=True, padding_idx=0, pooling_mode="gather"):
        super(LSTMClassifier, self).__init__(embedding_size, num_embeddings, num_classes, rnn_hidden_size, bidirectional, batch_first, padding_idx, pooling_mode)
        self.rnn = nn.LSTM(input_size=embedding_size,
                           hidden_size=rnn_hidden_size,
                           batch_first=batch_first,
                           bidirectional=bidirectional)

In [112]:
def set_seed_everywhere(seed, cuda):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if cuda:
        torch.cuda.manual_seed_all(seed)

def handle_dirs(dirpath):
    if not os.path.exists(dirpath):
        os.makedirs(dirpath)

### Settings

In [113]:
args = Namespace(
    # Data and path information
    csv_path = "src/tweets_cleaned.csv",
    output_dir = "src/sentiment_outputs",
    model_file = "sentiment_model.pth",
    # Model hyper parameter
    char_embedding_size=100,
    rnn_hidden_size=64,
    model_type="LSTM", #RNN, GRU, LSTM
    bidirectional=False, #True, False
    pooling_mode="gather", #mean, max, gather
    # Training hyper parameter
    seed=1337,
    num_epochs=50,
    batch_size=128,
    learning_rate=1e-3,
    early_stopping_criteria=5,
    # Runtime hyper parameter
    cuda=True,
    catch_keyboard_interrupt=True,
    expand_filepaths_to_output_dir=True,
)

# Check CUDA
if not torch.cuda.is_available():
    args.cuda = False

args.device = torch.device("cuda" if args.cuda else "cpu")

print("Using CUDA: {}".format(args.cuda))


if args.expand_filepaths_to_output_dir:
    args.model_state_file = os.path.join(args.output_dir, args.model_file)

# Set seed for reproducibility
set_seed_everywhere(args.seed, args.cuda)

# handle dirs
handle_dirs(args.output_dir)

Using CUDA: False


### Create Dataset and Vectorizer

In [114]:
dataset = SentimentDataset.load_dataset_and_make_vectorizer(args.csv_path)
vectorizer = dataset.get_vectorizer()

print(type(dataset.df['tokens'].iloc[0]))
print(dataset.df['tokens'].iloc[0])

if args.model_type == "RNN":
    classifier = RNNClassifier(
        embedding_size=args.char_embedding_size,
        num_embeddings=len(vectorizer.text_vocab),
        num_classes=len(vectorizer.label_vocab),
        rnn_hidden_size=args.rnn_hidden_size,
        padding_idx=vectorizer.text_vocab.mask_index,
        bidirectional=args.bidirectional,
        pooling_mode=args.pooling_mode
    )
elif args.model_type == "GRU":
    classifier = GRUClassifier(
        embedding_size=args.char_embedding_size,
        num_embeddings=len(vectorizer.text_vocab),
        num_classes=len(vectorizer.label_vocab),
        rnn_hidden_size=args.rnn_hidden_size,
        padding_idx=vectorizer.text_vocab.mask_index,
        bidirectional=args.bidirectional,
        pooling_mode=args.pooling_mode
    )
elif args.model_type == "LSTM":
    classifier = LSTMClassifier(
        embedding_size=args.char_embedding_size,
        num_embeddings=len(vectorizer.text_vocab),
        num_classes=len(vectorizer.label_vocab),
        rnn_hidden_size=args.rnn_hidden_size,
        padding_idx=vectorizer.text_vocab.mask_index,
        bidirectional=args.bidirectional,
        pooling_mode=args.pooling_mode
    )

<class 'list'>
['offer', 'group', 'pricing', 'discounts', 'chance']


In [115]:
dataset._max_seq_length

29

In [116]:
vectorizer.text_vocab._token_to_idx

{'<MASK>': 0,
 '<UNK>': 1,
 '<BEGIN>': 2,
 '<END>': 3,
 'said': 4,
 'plus': 5,
 'added': 6,
 'commercials': 7,
 'experience': 8,
 'tacky': 9,
 'not': 10,
 'today': 11,
 'must': 12,
 'mean': 13,
 'need': 14,
 'take': 15,
 'another': 16,
 'trip': 17,
 'really': 18,
 'aggressive': 19,
 'blast': 20,
 'obnoxious': 21,
 'entertainment': 22,
 'guests': 23,
 'faces': 24,
 'little': 25,
 'recourse': 26,
 'big': 27,
 'bad': 28,
 'thing': 29,
 'seriously': 30,
 'would': 31,
 'pay': 32,
 '30': 33,
 'flight': 34,
 'seats': 35,
 'playing': 36,
 'flying': 37,
 'va': 38,
 'yes': 39,
 'nearly': 40,
 'every': 41,
 'time': 42,
 'fly': 43,
 'vx': 44,
 'ear': 45,
 'worm': 46,
 'go': 47,
 'away': 48,
 'missed': 49,
 'prime': 50,
 'opportunity': 51,
 'men': 52,
 'without': 53,
 'hats': 54,
 'parody': 55,
 'well': 56,
 'amazing': 57,
 'arrived': 58,
 'hour': 59,
 'early': 60,
 'good': 61,
 'know': 62,
 'suicide': 63,
 'second': 64,
 'leading': 65,
 'death': 66,
 'among': 67,
 'teens': 68,
 '10': 69,
 '24': 70

In [117]:
vectorizer.label_vocab._token_to_idx

{'neutral': 0, 'positive': 1, 'negative': 2}

### Training Routine

In [118]:
def make_train_state(args):
    return {'stop_early': False,
            'early_stopping_step': 0,
            'early_stopping_best_val': 1e8,
            'learning_rate': args.learning_rate,
            'epoch_index': 0,
            'train_loss': [],
            'train_acc': [],
            'val_loss': [],
            'val_acc': [],
            'test_loss': -1,
            'test_acc': -1,
            'model_filename': args.model_state_file}


def update_train_state(args, model, train_state):
    """Handle the training state updates.

    Components:
     - Early Stopping: Prevent overfitting.
     - Model Checkpoint: Model is saved if the model is better

    :param args: main arguments
    :param model: model to train
    :param train_state: a dictionary representing the training state values
    :returns:
        a new train_state
    """

    # Save one model at least
    if train_state['epoch_index'] == 0:
        torch.save(model.state_dict(), train_state['model_filename'])
        train_state['stop_early'] = False

    # Save model if performance improved
    elif train_state['epoch_index'] >= 1:
        loss_tm1, loss_t = train_state['val_loss'][-2:]

        # If loss worsened
        #if loss_t >= loss_tm1: # if current loss becomes smaller than the previous loss, early_stopping_step is reset to 0.
                                # this statement makes the model training time longer, compared to the statement below.
        if loss_t >= train_state['early_stopping_best_val']:  # curent loss is compared with early_stopping_best_val loss value
            # Update step
            train_state['early_stopping_step'] += 1
        # Loss decreased
        else:
            # Save the best model
            if loss_t < train_state['early_stopping_best_val']:
                torch.save(model.state_dict(), train_state['model_filename'])
                train_state['early_stopping_best_val'] = loss_t

            # Reset early stopping step
            train_state['early_stopping_step'] = 0

        # Stop early ?
        train_state['stop_early'] = \
            train_state['early_stopping_step'] >= args.early_stopping_criteria

    return train_state


def compute_accuracy(y_pred, y_target):
    _, y_pred_indices = y_pred.max(dim=1)
    n_correct = torch.eq(y_pred_indices, y_target).sum().item()
    return n_correct / len(y_pred_indices) * 100

In [119]:
# Move classifier to device
if args.model_type == "RNN":
    classifier = RNNClassifier(embedding_size=args.char_embedding_size,
                               num_embeddings=len(vectorizer.text_vocab),
                               num_classes=len(vectorizer.label_vocab),
                               rnn_hidden_size=args.rnn_hidden_size,
                               padding_idx=vectorizer.text_vocab.mask_index,
                               bidirectional=args.bidirectional,
                               pooling_mode=args.pooling_mode)
elif args.model_type == "GRU":
    classifier = GRUClassifier(embedding_size=args.char_embedding_size,
                               num_embeddings=len(vectorizer.text_vocab),
                               num_classes=len(vectorizer.label_vocab),
                               rnn_hidden_size=args.rnn_hidden_size,
                               padding_idx=vectorizer.text_vocab.mask_index,
                               bidirectional=args.bidirectional,
                               pooling_mode=args.pooling_mode)
elif args.model_type == "LSTM":
    classifier = LSTMClassifier(embedding_size=args.char_embedding_size,
                                num_embeddings=len(vectorizer.text_vocab),
                                num_classes=len(vectorizer.label_vocab),
                                rnn_hidden_size=args.rnn_hidden_size,
                                padding_idx=vectorizer.text_vocab.mask_index,
                                bidirectional=args.bidirectional,
                                pooling_mode=args.pooling_mode)
else:
    raise ValueError(f"Unknown model_type: {args.model_type}")

classifier = classifier.to(args.device)
dataset.class_weights = dataset.class_weights.to(args.device)

# Loss, optimizer, scheduler
loss_func = nn.CrossEntropyLoss(dataset.class_weights)
optimizer = optim.Adam(classifier.parameters(), lr=args.learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, mode='min',
                                                 factor=0.1, patience=10)

# Train state
train_state = make_train_state(args)

# Progress bars
epoch_bar = tqdm(desc='training routine', total=args.num_epochs, position=0)

dataset.set_split('train')
train_bar = tqdm(desc='split=train', total=dataset.get_num_batches(args.batch_size),
                 position=1, leave=True)

dataset.set_split('val')
val_bar = tqdm(desc='split=val', total=dataset.get_num_batches(args.batch_size),
               position=1, leave=True)

# Training loop
try:
    for epoch_index in range(args.num_epochs):
        train_state['epoch_index'] = epoch_index

        # Training step
        dataset.set_split('train')
        batch_generator = generate_batches(dataset, batch_size=args.batch_size, device=args.device)
        running_loss, running_acc = 0.0, 0.0
        classifier.train()

        for batch_index, batch_dict in enumerate(batch_generator):
            optimizer.zero_grad()

            y_pred = classifier(x_in=batch_dict['x_data'], x_lengths=batch_dict['x_length'])
            loss = loss_func(y_pred, batch_dict['y_target'])
            running_loss += (loss.item() - running_loss) / (batch_index + 1)

            loss.backward()
            optimizer.step()

            acc_t = compute_accuracy(y_pred, batch_dict['y_target'])
            running_acc += (acc_t - running_acc) / (batch_index + 1)

            train_bar.set_postfix(loss=running_loss, acc=running_acc, epoch=epoch_index)
            train_bar.update()

        train_state['train_loss'].append(running_loss)
        train_state['train_acc'].append(running_acc)

        # Validation step
        dataset.set_split('val')
        batch_generator = generate_batches(dataset, batch_size=args.batch_size, device=args.device)
        running_loss, running_acc = 0.0, 0.0
        classifier.eval()

        with torch.no_grad():
            for batch_index, batch_dict in enumerate(batch_generator):
                y_pred = classifier(x_in=batch_dict['x_data'], x_lengths=batch_dict['x_length'])
                loss = loss_func(y_pred, batch_dict['y_target'])
                running_loss += (loss.item() - running_loss) / (batch_index + 1)

                acc_t = compute_accuracy(y_pred, batch_dict['y_target'])
                running_acc += (acc_t - running_acc) / (batch_index + 1)

                val_bar.set_postfix(loss=running_loss, acc=running_acc, epoch=epoch_index)
                val_bar.update()

        train_state['val_loss'].append(running_loss)
        train_state['val_acc'].append(running_acc)

        # Update train state and scheduler
        train_state = update_train_state(args=args, model=classifier, train_state=train_state)
        scheduler.step(train_state['val_loss'][-1])

        train_bar.reset()
        val_bar.reset()
        epoch_bar.update()

        if train_state['stop_early']:
            break

except KeyboardInterrupt:
    print("Exiting loop")

training routine:   0%|          | 0/50 [00:00<?, ?it/s]

split=train:   0%|          | 0/77 [00:00<?, ?it/s]

split=val:   0%|          | 0/16 [00:00<?, ?it/s]

In [120]:
acc = train_state['train_acc']
val_acc = train_state['val_acc']
loss = train_state['train_loss']
val_loss = train_state['val_loss']

epochs = list(range(1, len(acc) + 1))

# Create interactive line chart
fig = go.Figure()

for y, name, color, mode in [(loss, 'Training Loss', 'blue', 'lines+markers'),
                             (val_loss, 'Validation Loss', 'royalblue', 'lines')]:
    fig.add_trace(go.Scatter(x=epochs, y=y, mode=mode, name=name, line=dict(color=color)))

fig.update_layout(
    title='Training and Validation Loss',
    xaxis_title='Epochs',
    yaxis_title='Loss',
    legend=dict(x=0.02, y=0.98)
)

fig.show()

In [121]:
# Create interactive line chart for accuracy
fig = go.Figure()

for y, name, color, mode in [(acc, 'Training Accuracy', 'blue', 'lines+markers'),
                             (val_acc, 'Validation Accuracy', 'royalblue', 'lines')]:
    fig.add_trace(go.Scatter(x=epochs, y=y, mode=mode, name=name, line=dict(color=color)))

fig.update_layout(
    title='Training and Validation Accuracy',
    xaxis_title='Epochs',
    yaxis_title='Accuracy',
    legend=dict(x=0.02, y=0.98)
)

fig.show()

In [122]:
# compute the loss & accuracy on the test set using the best available model

classifier.load_state_dict(torch.load(train_state['model_filename'],weights_only=False))

classifier = classifier.to(args.device)
dataset.class_weights = dataset.class_weights.to(args.device)
loss_func = nn.CrossEntropyLoss(dataset.class_weights)

dataset.set_split('test')
batch_generator = generate_batches(dataset,
                                   batch_size=args.batch_size,
                                   device=args.device)
running_loss = 0.
running_acc = 0.
classifier.eval()

y_pred_list = []         # store predicted values for confusion matrix
y_label_list = []  # ground truth value

with torch.no_grad():  # Disable gradient computation
  for batch_index, batch_dict in enumerate(batch_generator):
    # compute the output
    y_pred =  classifier(batch_dict['x_data'],
                         x_lengths=batch_dict['x_length'])

    # store predicted values and ground truth values for calculating confusion matrix
    y_pred_list.extend(y_pred.max(dim=1)[1].cpu().numpy())
    y_label_list.extend(batch_dict['y_target'].cpu().numpy())

    # compute the loss
    loss = loss_func(y_pred, batch_dict['y_target'])
    loss_t = loss.item()
    running_loss += (loss_t - running_loss) / (batch_index + 1)

    # compute the accuracy
    acc_t = compute_accuracy(y_pred, batch_dict['y_target'])
    running_acc += (acc_t - running_acc) / (batch_index + 1)

train_state['test_loss'] = running_loss
train_state['test_acc'] = running_acc

In [123]:
print("Test loss: {};".format(train_state['test_loss']))
print("Test Accuracy: {}".format(train_state['test_acc']))

Test loss: 0.791440986096859;
Test Accuracy: 71.38671875


In [124]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
label_classes = []
for i in range(len(dataset._vectorizer.label_vocab)):
    label_classes.append(dataset._vectorizer.label_vocab.lookup_index(i))
print(label_classes)

['neutral', 'positive', 'negative']


In [125]:
import pandas as pd
cm = confusion_matrix(y_label_list, y_pred_list)
cm_df = pd.DataFrame(cm.T, index=label_classes, columns=label_classes)
cm_df.index.name = 'Predicted'
cm_df.columns.name = 'True'
print(cm_df)

True       neutral  positive  negative
Predicted                             
neutral        178        39       147
positive       119       234       113
negative       126        42      1050


In [126]:
print(classification_report(y_label_list, y_pred_list))

              precision    recall  f1-score   support

           0       0.49      0.42      0.45       423
           1       0.50      0.74      0.60       315
           2       0.86      0.80      0.83      1310

    accuracy                           0.71      2048
   macro avg       0.62      0.66      0.63      2048
weighted avg       0.73      0.71      0.72      2048

